# GPT Action 库：Gmail

## 简介

本页面为开发者针对特定应用构建 GPT Action 提供说明和指南。在继续之前，请先熟悉以下内容：
- [GPT Actions 简介](https://platform.openai.com/docs/actions)
- [GPT Actions 库简介](https://platform.openai.com/docs/actions/actions-library)
- [从零开始构建 GPT Action 的示例](https://platform.openai.com/docs/actions/getting-started)

本 GPT Action 概述了如何连接 Google Gmail——Google 面向个人或企业的私密、安全的电子邮件服务。此 Action 连接到 Google Gmail API，可以在已授权的账户中读取、发送、列出和起草电子邮件。

### 价值与业务用例示例

**价值**：Gmail GPT 将成为一个强大的工具，用于简化沟通流程、提升客户互动并优化资源分配。

**用例示例**：
- 管理内部沟通：总结冗长的电子邮件，并根据之前的邮件往来起草回复。
- 客服人员可以按照公司的沟通准则、语气和风格，为客户提供即时回复。
- 引用其他 GPT（例如数据分析 GPT），然后要求通过电子邮件起草或发送汇总后的分析。

## 应用信息

### 应用关键链接

开始之前，请先查看该应用的以下链接：
- 应用网站：https://mail.google.com/mail/u/0/#inbox
- 应用 API 文档：https://developers.google.com/gmail/api/guides

### 应用前提条件

开始之前，请确保你拥有 Google Cloud 账户，并已启用 Gmail API：
- 设置一个 Google Cloud 项目
- 从 Google API Library 启用 Gmail API
- 如果应用的 “Publishing Status” 为 “Testing”，请确保已将用户添加到你的应用中

## ChatGPT 步骤

### 自定义 GPT 指令

创建自定义 GPT 后，将下面的文本复制到 Instructions 面板中。有疑问？请查看[入门示例](https://platform.openai.com/docs/actions/getting-started)，进一步了解此步骤的详细工作方式。

In [ ]:
**Context**
Act as an email assistant designed to enhance user interaction with emails in various ways. This GPT can assist with productivity by summarizing emails/threads, identifying next steps/follow-ups, drafting or sending pre-written responses, and programmatically interacting with third-party tools (e.g., Notion to-dos, Slack channel summaries, data extraction for responses). This GPT has full scope access to the GMAIL OAuth 2.0 API, capable of reading, composing, sending, and permanently deleting emails from Gmail.

**Instructions**
- Always conclude an email by signing off with logged in user's name, unless otherwise stated.
- Verify that the email data is correctly encoded in the required format (e.g., base64 for the message body).
- Email Encoding Process: 1\ Construct the email message in RFC 2822 format. 2\ Base64 encode the email message. 3\Send the encoded message using the API.
- If not specified, sign all emails with the user name.
- API Usage: After answering the user's question, do not call the Google API again until another question is asked.
- All emails created, draft or sent, should be in plain text.
- Ensure that the email format is clean and is formatted as if someone sent the email from their own inbox. Once a draft is created or email sent, display a message to the user confirming that the draft is ready or the email is sent.
- Check that the "to" email address is valid and in the correct format. It should be in the format "recipient@example.com". 
- Only provide summaries of existing emails; do not fabricate email content.
- Professionalism: Behave professionally, providing clear and concise responses.
- Clarification: Ask for clarification when needed to ensure accuracy and completeness in fulfilling user requests.
- Privacy and Security: Respect user privacy and handle all data securely.


### OpenAPI Schema

创建自定义 GPT 后，将下面的文本复制到 Actions 面板中。有疑问？请查看[入门示例](https://platform.openai.com/docs/actions/getting-started)，进一步了解此步骤的详细工作方式。

In [ ]:
openapi: 3.1.0

info:
  title: Gmail Email API
  version: 1.0.0
  description: API to read, write, and send emails in a Gmail account.

servers:
  - url: https://gmail.googleapis.com

paths:
  /gmail/v1/users/{userId}/messages:
    get:
      summary: List All Emails
      description: Lists all the emails in the user's mailbox.
      operationId: listAllEmails
      parameters:
        - name: userId
          in: path
          required: true
          schema:
            type: string
          description: The user's email address. Use "me" to indicate the authenticated user.
        - name: q
          in: query
          schema:
            type: string
          description: Query string to filter messages (optional).
        - name: pageToken
          in: query
          schema:
            type: string
          description: Token to retrieve a specific page of results in the list.
        - name: maxResults
          in: query
          schema:
            type: integer
            format: int32
          description: Maximum number of messages to return.
      responses:
        '200':
          description: Successful response
          content:
            application/json:
              schema:
                $ref: '#/components/schemas/MessageList'
        '400':
          description: Bad Request
        '401':
          description: Unauthorized
        '403':
          description: Forbidden
        '404':
          description: Not Found
        '500':
          description: Internal Server Error

  /gmail/v1/users/{userId}/messages/send:
    post:
      summary: Send Email
      description: Sends a new email.
      operationId: sendEmail
      parameters:
        - name: userId
          in: path
          required: true
          schema:
            type: string
          description: The user's email address. Use "me" to indicate the authenticated user.
      requestBody:
        required: true
        content:
          application/json:
            schema:
              $ref: '#/components/schemas/Message'
      responses:
        '200':
          description: Email sent successfully
          content:
            application/json:
              schema:
                $ref: '#/components/schemas/Message'
        '400':
          description: Bad Request
        '401':
          description: Unauthorized
        '403':
          description: Forbidden
        '500':
          description: Internal Server Error

  /gmail/v1/users/{userId}/messages/{id}:
    get:
      summary: Read Email
      description: Gets the full email content including headers and body.
      operationId: readEmail
      parameters:
        - name: userId
          in: path
          required: true
          schema:
            type: string
          description: The user's email address. Use "me" to indicate the authenticated user.
        - name: id
          in: path
          required: true
          schema:
            type: string
          description: The ID of the email to retrieve.
      responses:
        '200':
          description: Successful response
          content:
            application/json:
              schema:
                $ref: '#/components/schemas/FullMessage'
        '400':
          description: Bad Request
        '401':
          description: Unauthorized
        '403':
          description: Forbidden
        '404':
          description: Not Found
        '500':
          description: Internal Server Error

  /gmail/v1/users/{userId}/messages/{id}/modify:
    post:
      summary: Modify Label
      description: Modify labels of an email.
      operationId: modifyLabels
      parameters:
        - name: userId
          in: path
          required: true
          schema:
            type: string
          description: The user's email address. Use "me" to indicate the authenticated user.
        - name: id
          in: path
          required: true
          schema:
            type: string
          description: The ID of the email to change labels.
      requestBody:
        required: true
        content:
          application/json:
            schema:
              $ref: '#/components/schemas/LabelModification'
      responses:
        '200':
          description: Labels modified successfully
          content:
            application/json:
              schema:
                $ref: '#/components/schemas/Message'
        '400':
          description: Bad Request
        '401':
          description: Unauthorized
        '403':
          description: Forbidden
        '500':
          description: Internal Server Error

  /gmail/v1/users/{userId}/drafts:
    post:
      summary: Create Draft
      description: Creates a new email draft.
      operationId: createDraft
      parameters:
        - name: userId
          in: path
          required: true
          schema:
            type: string
          description: The user's email address. Use "me" to indicate the authenticated user.
      requestBody:
        required: true
        content:
          application/json:
            schema:
              $ref: '#/components/schemas/Draft'
      responses:
        '200':
          description: Draft created successfully
          content:
            application/json:
              schema:
                $ref: '#/components/schemas/Draft'
        '400':
          description: Bad Request
        '401':
          description: Unauthorized
        '403':
          description: Forbidden
        '500':
          description: Internal Server Error

  /gmail/v1/users/{userId}/drafts/send:
    post:
      summary: Send Draft
      description: Sends an existing email draft.
      operationId: sendDraft
      parameters:
        - name: userId
          in: path
          required: true
          schema:
            type: string
          description: The user's email address. Use "me" to indicate the authenticated user.
      requestBody:
        required: true
        content:
          application/json:
            schema:
              $ref: '#/components/schemas/SendDraftRequest'
      responses:
        '200':
          description: Draft sent successfully
          content:
            application/json:
              schema:
                $ref: '#/components/schemas/Message'
        '400':
          description: Bad Request
        '401':
          description: Unauthorized
        '403':
          description: Forbidden
        '500':
          description: Internal Server Error

components:
  schemas:
    MessageList:
      type: object
      properties:
        messages:
          type: array
          items:
            $ref: '#/components/schemas/Message'
        nextPageToken:
          type: string

    Message:
      type: object
      properties:
        id:
          type: string
        threadId:
          type: string
        labelIds:
          type: array
          items:
            type: string
        addLabelIds:
          type: array
          items:
            type: string
        removeLabelIds:
          type: array
          items:
            type: string
        snippet:
          type: string
        raw:
          type: string
          format: byte
          description: The entire email message in an RFC 2822 formatted and base64url encoded string.

    FullMessage:
      type: object
      properties:
        id:
          type: string
        threadId:
          type: string
        labelIds:
          type: array
          items:
            type: string
        snippet:
          type: string
        payload:
          type: object
          properties:
            headers:
              type: array
              items:
                type: object
                properties:
                  name:
                    type: string
                  value:
                    type: string
            parts:
              type: array
              items:
                type: object
                properties:
                  mimeType:
                    type: string
                  body:
                    type: object
                    properties:
                      data:
                        type: string

    LabelModification:
      type: object
      properties:
        addLabelIds:
          type: array
          items:
            type: string
        removeLabelIds:
          type: array
          items:
            type: string

    Label:
      type: object
      properties:
        addLabelIds:
          type: array
          items:
            type: string
        removeLabelIds:
          type: array
          items:
            type: string

    EmailDraft:
      type: object
      properties:
        to:
          type: array
          items:
            type: string
        cc:
          type: array
          items:
            type: string
        bcc:
          type: array
          items:
            type: string
        subject:
          type: string
        body:
          type: object
          properties:
            mimeType:
              type: string
              enum: [text/plain, text/html]
            content:
              type: string

    Draft:
      type: object
      properties:
        id:
          type: string
        message:
          $ref: '#/components/schemas/Message'

    SendDraftRequest:
      type: object
      properties:
        draftId:
          type: string
          description: The ID of the draft to send.
        userId:
          type: string
          description: The user's email address. Use "me" to indicate the authenticated user.

## 身份验证说明

以下是为该第三方应用设置身份验证的说明。有疑问？请查看[入门示例](https://platform.openai.com/docs/actions/getting-started)，进一步了解此步骤的详细工作方式。

### Action 前置步骤

在 ChatGPT 中设置身份验证之前，请先在应用中完成以下步骤。


- 前往 Google Cloud Console
- 导航到 API & Services > Credentials

![gptactions_BigQuery_auth.png](../../../images/gptactions_Gmail_enableAPIs.png)

![gptactions_BigQuery_auth.png](../../../images/gptactions_Gmail_gmailApiTile.png)

- 创建新的 OAuth 凭据（或使用已有的凭据）

![gptactions_BigQuery_auth.png](../../../images/gptactions_Gmail_apikey.png)

- 找到你的 OAuth Client ID 和 Client Secret，并安全地保存这两个值（参见下方截图）

![gptactions_BigQuery_auth.png](../../../images/gptactions_Gmail_clientidsecret.png)

### 在 ChatGPT 中

在 ChatGPT 中，点击 "Authentication" 并选择 **"OAuth"**。填写以下信息。

- **Client ID**：使用上述步骤中的 Client ID
- **Client Secret**：使用上述步骤中的 Client Secret
- **Authorization URL**：https://accounts.google.com/o/oauth2/auth
- **Token URL**：https://oauth2.googleapis.com/token
- **Scope**：https://mail.google.com/
- **Token**：Default (POST)

### Action 后续步骤

在 ChatGPT 中设置好身份验证后，请在应用中按照以下步骤完成 Action 的最终配置。

- 从 GPT Action 中复制回调 URL
- 在 “Authorized redirect URIs” 中（参见上方截图）添加你的回调 URL


### 常见问题与故障排除

- *回调 URL 错误：* 如果你在 ChatGPT 中遇到回调 URL 错误，请仔细查看上面的截图。你需要将回调 URL 直接添加到 GCP 中，Action 才能正确完成身份验证
- *Schema 调用了错误的项目或数据集：* 如果 ChatGPT 调用了错误的项目或数据集，请考虑更新你的指令，使其更加明确：(a) 应该调用哪个项目/数据集，或者 (b) 要求用户在运行查询之前提供这些确切信息

*有希望我们优先支持的集成吗？我们的集成中有错误吗？请在我们的 github 上提交 PR 或 issue，我们会查看处理。*
